<a href="https://colab.research.google.com/github/deepanrajm/GL/blob/main/LLM/BERT_Classify_different_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

%pip install -U huggingface_hub

# To get an HF_TOKEN:
# 1. Go to https://huggingface.co/settings/tokens
# 2. Generate a new token with 'read' role (or 'write' if you plan to upload models)
# 3. In Colab, go to the 'Secrets' tab (🔑 icon on the left panel)
# 4. Add a new secret named `HF_TOKEN` and paste your token value there.
#    Make sure 'Notebook access' is enabled for this secret.

from huggingface_hub import login
from google.colab import userdata

try:
    # Attempt to retrieve the HF_TOKEN from Colab secrets
    hf_token = userdata.get('HF_TOKEN')
    login(token=hf_token)
    print("Successfully logged in to Hugging Face Hub.")
except Exception as e:
    print(f"Could not log in to Hugging Face Hub. Please ensure HF_TOKEN is set in Colab secrets. Error: {e}")
    print("You may continue without a token, but might encounter rate limits or access issues.")


In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from datasets import load_dataset
import numpy as np
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score


model_name = "roberta-base" # try: "bert-base-uncased", "albert-base-v2", "microsoft/deberta-v3-base"


tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)


ds = load_dataset("stanfordnlp/imdb")


def tok(batch):
    return tokenizer(batch["text"], truncation=True, padding="max_length", max_length=256)


encoded = ds.map(tok, batched=True)
encoded = encoded.rename_column("label", "labels")
encoded.set_format("torch", columns=["input_ids","attention_mask","labels"])


args = TrainingArguments(
"roberta-imdb",
learning_rate=2e-5, num_train_epochs=3, per_device_train_batch_size=16,
per_device_eval_batch_size=32, weight_decay=0.01, eval_strategy="epoch",
fp16=True,
report_to="none"
)


def compute_metrics(p):
    preds = np.argmax(p.predictions, axis=1)
    return {
    "acc": accuracy_score(p.label_ids, preds),
    "f1": f1_score(p.label_ids, preds, average="macro"),
    "prec": precision_score(p.label_ids, preds, average="macro"),
    "rec": recall_score(p.label_ids, preds, average="macro"),
    }


trainer = Trainer(model=model, args=args, train_dataset=encoded["train"], eval_dataset=encoded["test"], compute_metrics=compute_metrics)
trainer.train()